# THỰC HÀNH BUỔI 5: SPARK DATAFRAME & EXECUTION PLAN

Chào mừng bạn đến với buổi học số 5. Trong buổi này, chúng ta sẽ học về:
- **Schema**: Cách Spark quản lý cấu trúc dữ liệu.
- **DataFrame API**: `select`, `filter` (hoặc `where`), `withColumn`, `cast`, `drop`, `dropDuplicates`.
- **Catalyst Optimizer**: Trình tối ưu hóa truy vấn của Spark SQL.
- **Execution Plan**: Học cách đọc kế hoạch thực thi bằng phương thức `explain()`.

## 1. LÝ THUYẾT VỀ CATALYST OPTIMIZER

**Catalyst Optimizer** là lõi tối ưu hóa của Spark SQL, chịu trách nhiệm chuyển đổi câu lệnh (SQL hoặc DataFrame API) thành một tập hợp các chỉ thị thực thi vật lý tối ưu nhất trên cụm Cluster. Quá trình này trải qua 4 giai đoạn chính:

```
 [ DataFrame / SQL ]
         │
         ▼ (1. Analysis)
 [ Parsed Logical Plan ]  ──> Kiểm tra Catalog (bảng, cột, kiểu dữ liệu)
         │
         ▼ (2. Logical Optimization)
 [ Optimized Logical Plan ] ──> Áp dụng các quy tắc (Filter Pushdown, Projection Pruning...)
         │
         ▼ (3. Physical Planning)
 [ Physical Plans ] ──> Đánh giá chi phí (Cost Model) để chọn plan tốt nhất
         │
         ▼ (4. Code Generation)
 [ Java Bytecode ] (Tạo mã nguồn Java chạy trực tiếp trên JVM của Executors)
```

### Các kỹ thuật tối ưu hóa phổ biến của Catalyst:
1. **Predicate Pushdown (Filter Pushdown)**: Đẩy bộ lọc điều kiện (ví dụ: `status = 'delivered'`) xuống sát nguồn dữ liệu nhất có thể. Thay vì đọc toàn bộ file rồi lọc, Spark sẽ lọc ngay lúc đọc hoặc yêu cầu nguồn dữ liệu (như Parquet, Database) chỉ trả về những bản ghi thỏa mãn điều kiện.
2. **Projection Pruning (Column Pruning)**: Chỉ đọc các cột thực sự được dùng trong câu lệnh và loại bỏ các cột thừa. Việc này giúp tiết kiệm I/O mạng và bộ nhớ rất nhiều.
3. **Constant Folding**: Tính toán trước các biểu thức hằng số (ví dụ: `1 + 1` chuyển trực tiếp thành `2` trước khi chạy).
4. **Project / Filter Collapse**: Gộp nhiều bước Project (chọn cột) hoặc Filter liên tiếp thành một bước duy nhất.

## Khởi tạo Spark Session

Chúng ta sẽ khởi chạy Spark ở chế độ `local[*]` (sử dụng tất cả các nhân CPU trên máy cục bộ của bạn).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Khởi tạo Spark Session cục bộ
spark = SparkSession.builder \
    .appName("Spark DataFrame Practice Session 5") \
    .master("local[*]") \
    .getOrCreate()

# Thiết lập log level sang WARN để hạn chế log thông tin thừa
spark.sparkContext.setLogLevel("WARN")
print("Spark Session đã khởi tạo thành công!")

## BÀI 1: Đọc file `olist_orders_dataset.csv` và in Schema

- Spark hỗ trợ đọc nhiều định dạng file khác nhau. Ở đây ta đọc file CSV.
- Tham số `header=True` báo cho Spark dòng đầu tiên của file là tên cột.
- Tham số `inferSchema=True` yêu cầu Spark tự quét dữ liệu để phỏng đoán kiểu dữ liệu của từng cột (như String, Integer, Timestamp, v.v.).
- Phương thức `.printSchema()` dùng để in cấu trúc bảng DataFrame.

In [ ]:
# Đường dẫn tới file orders.csv trong thư mục data/raw
import os
csv_path = os.path.join("..", "data", "raw", "olist_orders_dataset.csv")

# Đọc dữ liệu
df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)

# In Schema
df_raw.printSchema()

# Hiển thị 5 dòng đầu tiên để xem cấu trúc thực tế
df_raw.show(5, truncate=False)

## BÀI 2: Chỉ lấy `order_id`, `customer_id`, `purchase_timestamp`, `status`

- Trong file CSV gốc, các cột này được đặt tên lần lượt là `order_id`, `customer_id`, `order_purchase_timestamp`, và `order_status`.
- Chúng ta sử dụng `.select()` để lấy các cột mong muốn, đồng thời dùng `.alias()` để đổi tên hiển thị tương ứng.

In [ ]:
# Lấy cột và đặt tên bí danh (alias) tương ứng với yêu cầu đề bài
df_selected = df_raw.select(
    col("order_id"),
    col("customer_id"),
    col("order_purchase_timestamp").alias("purchase_timestamp"),
    col("order_status").alias("status")
)

df_selected.show(5, truncate=False)

## BÀI 3: Đổi `purchase_timestamp` thành `timestamp` và chuyển đổi kiểu dữ liệu (Cast)

- Chúng ta sử dụng `.withColumnRenamed(old_name, new_name)` để đổi tên cột từ `purchase_timestamp` thành `timestamp`.
- Chúng ta sử dụng `.withColumn(col_name, col_expr)` kết hợp với `.cast("datatype")` để ép kiểu dữ liệu cột sang định dạng `timestamp` (nếu chưa phải).

In [ ]:
# Đổi tên cột và ép kiểu dữ liệu về timestamp
df_casted = df_selected \
    .withColumnRenamed("purchase_timestamp", "timestamp") \
    .withColumn("timestamp", col("timestamp").cast("timestamp"))

# In lại schema để kiểm tra kiểu dữ liệu của cột timestamp mới
df_casted.printSchema()
df_casted.show(5, truncate=False)

## BÀI 4: Chỉ giữ lại các đơn hàng có trạng thái là `delivered`

- Chúng ta sử dụng phương thức `.filter(condition)` (hoặc `.where(condition)`) để lọc các dòng dữ liệu thoả mãn điều kiện.

In [ ]:
# Lọc dòng theo trạng thái status
df_filtered = df_casted.filter(col("status") == "delivered")

df_filtered.show(5, truncate=False)
print(f"Tổng số đơn hàng delivered: {df_filtered.count()}")

## BÀI 5: Xem Execution Plan (Kế hoạch thực thi)

- Spark là một công cụ tính toán trì hoãn (**Lazy Evaluation**). Khi chúng ta gọi các phép transform như `select`, `filter`, `withColumn`, Spark chưa hề chạy tính toán dữ liệu thật sự. Nó chỉ ghi nhận các bước này vào một đồ thị có hướng gọi là **DAG (Directed Acyclic Graph)**.
- Chỉ khi gặp một **Action** (ví dụ: `show()`, `count()`, `collect()`, `write()`), Spark mới kích hoạt Catalyst Optimizer để tối ưu hóa DAG này và tạo ra mã thực thi vật lý.
- Phương thức `.explain(True)` cho phép chúng ta xem toàn bộ quá trình biến đổi kế hoạch thực thi từ lúc khai báo cho tới khi biên dịch vật lý.

In [ ]:
# In chi tiết kế hoạch thực thi ra màn hình
df_filtered.explain(True)

### GIẢI THÍCH CHI TIẾT CÁC BƯỚC EXECUTION PLAN:

Khi đọc kết quả của `.explain(True)`, bạn sẽ thấy 4 phần kế hoạch khác nhau (từ trên xuống dưới):

#### 1. == Parsed Logical Plan ==
- Đây là bước đầu tiên ngay sau khi Spark nhận câu lệnh từ bạn. Spark phân tích cú pháp chuỗi câu lệnh và chuyển đổi thành một cây cấu trúc dữ liệu nội bộ.
- Các phép biến đổi được liệt kê đúng theo trình tự bạn viết code (Ví dụ: `Filter` -> `Project (cast)` -> `Project (rename)` -> `Project (select/alias)` -> `Relation (csv)`).
- Lúc này Spark chưa kiểm tra xem tên cột hay tên bảng có thực sự tồn tại trong nguồn dữ liệu hay không.

#### 2. == Analyzed Logical Plan ==
- Ở bước này, Spark truy vấn **Catalog** (metadata lưu trữ cấu trúc bảng/cột) để đối chiếu.
- Nó kiểm tra xem các cột `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp` có thực sự tồn tại trong file CSV không, đồng thời phân giải các kiểu dữ liệu của cột.
- Kế hoạch này trông khá giống Parsed Plan nhưng các kiểu dữ liệu và tên cột đã được định vị chính xác (gắn ID cụ thể như `order_id#17`).

#### 3. == Optimized Logical Plan ==
- **Đây là nơi ma thuật của Catalyst Optimizer xảy ra!** Catalyst sẽ áp dụng hàng loạt quy tắc để tối giản kế hoạch truy vấn.
- Bạn có thể nhận thấy điều kỳ diệu:
  1. **Project Collapse**: Cả 3 bước `Project` (chọn cột, rename, cast) lồng nhau ở trên đã được Catalyst gộp thành **1 bước Project duy nhất**: `Project [order_id#17, customer_id#18, order_purchase_timestamp#20 AS timestamp#46, order_status#19 AS status#26]`. Phép cast kiểu dữ liệu cũng được tối giản vì dữ liệu gốc đọc vào đã được suy diễn đúng kiểu `timestamp`.
  2. **Filter Pushdown (Predicate Pushdown)**: Bộ lọc `(order_status#19 = delivered)` đã được đẩy xuống ngay phía trên nguồn đọc CSV, đồng thời tự động thêm điều kiện lọc các giá trị Null: `(isnotnull(order_status#19) AND (order_status#19 = delivered))`.

#### 4. == Physical Plan ==
- Đây là kế hoạch thực tế sẽ chạy trên các JVM CPU cores.
- Kế hoạch này chứa các toán tử vật lý như `*(1) Filter`, `*(1) Project`, và `FileScan csv`.
- Ký hiệu `*(1)` biểu thị kỹ thuật **Whole-Stage Code Generation**: Spark gộp nhiều toán tử vật lý lại để tạo ra một đoạn code Java duy nhất, biên dịch trực tiếp sang Bytecode để thực thi trong một vòng lặp lớn (Single Loop), loại bỏ việc chuyển giao dữ liệu qua lại giữa các bước trung gian trong bộ nhớ, tối ưu CPU cache tối đa.
- Hãy xem thông số `PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered)]` và `ReadSchema`. Spark chỉ đọc 4 cột cần dùng từ file CSV thay vì đọc hết cả 8 cột của file gốc. Đây chính là **Schema Pruning** và **Predicate Pushdown** ở mức vật lý nguồn dữ liệu!

### Dọn dẹp tài nguyên

Sau khi hoàn thành thực hành, chúng ta dừng Spark Session để giải phóng RAM và CPU.

In [ ]:
spark.stop()
print("Spark Session đã dừng thành công!")